# Libraries

In [ ]:
import re
import sys
import warnings
warnings.filterwarnings("ignore")

import camelot
import numpy as np
import pandas as pd
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import ROOT_PATH, DATA_PATH, PROCESSED_PATH, EXTERNAL_PATH
from tfm.geo import (
    normalize_mun_code, get_prov_code,
    BCN_DISTRICT_MUN_CODES, MAD_DISTRICT_MUN_CODES, CP_TO_CUSTOM_INE,
)

np.random.seed(42)

---
# Municipios

In [ ]:
# municipios.csv incluye filas de distritos con formato 'BCN - Ciutat Vella' y 'MAD - Centro'
df_municipios = pd.read_csv(DATA_PATH / "external" / "municipios.csv", encoding="utf-8", low_memory=False)

print(df_municipios.shape)
display(df_municipios.head())

In [ ]:
# Separar mun_name en municipio y distrito para las filas de BCN y MAD
is_bcn = df_municipios["mun_name"].str.startswith("BCN - ", na=False)
is_mad = df_municipios["mun_name"].str.startswith("MAD - ", na=False)

df_municipios["district_name"] = pd.NA

df_municipios.loc[is_bcn, "district_name"] = (
    df_municipios.loc[is_bcn, "mun_name"].str.replace("BCN - ", "", regex=False)
)
df_municipios.loc[is_mad, "district_name"] = (
    df_municipios.loc[is_mad, "mun_name"].str.replace("MAD - ", "", regex=False)
)

df_municipios.loc[is_bcn, "mun_name"] = "Barcelona"
df_municipios.loc[is_mad, "mun_name"] = "Madrid"

# Eliminar filas de totales de ciudad (no son un distrito concreto)
df_municipios = df_municipios[df_municipios["district_name"] != "(General)"].copy()

# Probabilidad de tener coche: conductores por habitante
df_municipios["prob_car"] = (df_municipios["drivers"] / df_municipios["total_people"]).round(3)

display(df_municipios[df_municipios["mun_name"].isin(["Barcelona", "Madrid"])][["mun_code", "mun_name", "district_name", "prob_car"]])

In [ ]:
df_municipios["prov_code"]     = df_municipios["mun_code"].apply(get_prov_code)
df_municipios["mun_code_base"] = df_municipios["mun_code"].apply(normalize_mun_code)

check = df_municipios[df_municipios["mun_name"].isin(["Barcelona", "Madrid"])]
display(check[["mun_code", "district_name", "prov_code"]].head(4))


In [ ]:
df_municipios.to_csv(PROCESSED_PATH / "mun_enriched.csv", index=False)

print(f"Guardado: mun_enriched.csv  →  {df_municipios.shape}")

---
# Código Postal → Municipio

In [ ]:
df_cp_mun = pd.read_csv(EXTERNAL_PATH / "CP_CMUN_db.csv", encoding="utf-8", sep=";", low_memory=False)

df_cp_mun["cp_code"] = (
    pd.to_numeric(df_cp_mun["cp_code"], errors="coerce")
    .apply(lambda x: str(int(x)).zfill(5) if pd.notna(x) else pd.NA)
    .astype("string")
)
df_cp_mun["mun_code"] = pd.to_numeric(df_cp_mun["mun_code"], errors="coerce")

# Sobreescribir mun_code de Madrid y Barcelona con el código de distrito
df_cp_mun["mun_code"] = (
    df_cp_mun["cp_code"].map(CP_TO_CUSTOM_INE).fillna(df_cp_mun["mun_code"]).astype("Int64")
)

print(df_cp_mun.shape)
display(df_cp_mun.head())

In [ ]:
df_cp_mun.to_csv(PROCESSED_PATH / "cp_municipio.csv", index=False)
print(f"Guardado: cp_municipio.csv  -  {df_cp_mun.shape}")

---
# Barcelona

## Nivel de Estudios

In [ ]:
df_bcn_edu_raw = pd.read_csv(
    EXTERNAL_PATH / "BCN_nivel_estudios.csv", encoding="utf-8", sep=",", low_memory=False
)
df_bcn_edu_labels = pd.read_csv(
    EXTERNAL_PATH / "BCN_niveles_equival.csv", encoding="utf-8", sep=",", low_memory=False
)

print(df_bcn_edu_raw.shape)
display(df_bcn_edu_raw.head())

In [ ]:
df_bcn_edu_raw["mun_name"]      = "Barcelona"
df_bcn_edu_raw["district_name"] = df_bcn_edu_raw["Nom_Districte"].astype(str).str.strip()

# Usar códigos de municipios.csv
df_bcn_edu_raw["mun_code"] = df_bcn_edu_raw["district_name"].map(BCN_DISTRICT_MUN_CODES)

df_bcn_edu_raw["Valor"] = pd.to_numeric(df_bcn_edu_raw["Valor"], errors="coerce")
df_bcn_edu_raw = df_bcn_edu_raw.dropna(subset=["Valor", "mun_code"])

edu_labels = (
    df_bcn_edu_labels
    .loc[df_bcn_edu_labels["Desc_Dimensio"].str.contains("NIV", case=False, na=False),
         ["Codi_Valor", "Desc_Valor_ES"]]
    .copy()
    .rename(columns={"Codi_Valor": "NIV_EDUCA_esta", "Desc_Valor_ES": "education_level"})
)
edu_labels["NIV_EDUCA_esta"]     = pd.to_numeric(edu_labels["NIV_EDUCA_esta"],     errors="coerce")
df_bcn_edu_raw["NIV_EDUCA_esta"] = pd.to_numeric(df_bcn_edu_raw["NIV_EDUCA_esta"], errors="coerce")

df_bcn_edu_raw = df_bcn_edu_raw.merge(edu_labels, on="NIV_EDUCA_esta", how="left")

df_bcn_education = (
    df_bcn_edu_raw
    .groupby(["mun_code", "mun_name", "district_name", "NIV_EDUCA_esta", "education_level"], as_index=False)["Valor"]
    .sum()
)

df_bcn_education["probability"] = (
    df_bcn_education["Valor"]
    / df_bcn_education.groupby("district_name")["Valor"].transform("sum")
)

df_bcn_education = df_bcn_education.sort_values(["district_name", "NIV_EDUCA_esta"]).reset_index(drop=True)

print(df_bcn_education.shape)
display(df_bcn_education.head())

In [ ]:
df_bcn_education.to_csv(PROCESSED_PATH / "bcn_education.csv", index=False)

print(f"Guardado: bcn_education.csv  →  {df_bcn_education.shape}")

## Tamaño del Hogar

In [ ]:
df_bcn_hogar_raw = pd.read_csv(
    EXTERNAL_PATH / "BCN_tamaño_hogar.csv", encoding="utf-8", sep=",", low_memory=False
)

print(df_bcn_hogar_raw.shape)
display(df_bcn_hogar_raw.head())

In [ ]:
df_bcn_hogar_raw["mun_name"] = "Barcelona"

df_bcn_hogar_raw["district_name"] = np.where(
    df_bcn_hogar_raw["Tipo de territorio"] == "Districte",
    df_bcn_hogar_raw["Territorio"],
    np.nan
)

# Usar códigos de municipios.csv directamente
df_bcn_hogar_raw["mun_code"] = df_bcn_hogar_raw["district_name"].map(BCN_DISTRICT_MUN_CODES)

df_bcn_household = (
    df_bcn_hogar_raw[df_bcn_hogar_raw["district_name"].notna()]
    .copy()
    .rename(columns={"Número de personas del domicilio": "household_size", "01 ene 2025": "count"})
    [["mun_code", "mun_name", "district_name", "household_size", "count"]]
    .reset_index(drop=True)
)

print(df_bcn_household.shape)
display(df_bcn_household.head(10))

In [ ]:
df_bcn_household.to_csv(PROCESSED_PATH / "bcn_household.csv", index=False)

print(f"Guardado: bcn_household.csv  →  {df_bcn_household.shape}")

---
# Madrid

## Nivel de Estudios

In [ ]:
df_mad_education = pd.read_excel(EXTERNAL_PATH / "MAD_nivel_estudios.xlsx")

print(df_mad_education.shape)
display(df_mad_education.head())

## Tamaño del Hogar

In [ ]:
df_mad_hogar_raw = pd.read_excel(
    EXTERNAL_PATH / "Censo_Viviendas_Madrid.xlsx", sheet_name="aux_tfm"
)

df_mad_hogar_raw = df_mad_hogar_raw.dropna(how="all")
df_mad_hogar_raw = df_mad_hogar_raw.rename(columns={"Unnamed: 0": "Distrito"})
df_mad_hogar_raw = df_mad_hogar_raw.reset_index(drop=True)

cols_hogar = ["1 persona", "2 personas", "3 personas", "4 personas", "5 o más personas"]

df_mad_household = df_mad_hogar_raw.copy()
df_mad_household[cols_hogar] = df_mad_household[cols_hogar].div(df_mad_household["Total"], axis=0)
df_mad_household = df_mad_household[["Distrito"] + cols_hogar]

df_mad_household[["district_num", "district_name"]] = df_mad_household["Distrito"].str.extract(r"(\d+)\s*-\s*(.*)")
df_mad_household["mun_name"] = "Madrid"
df_mad_household["mun_code"] = df_mad_household["district_name"].str.strip().map(MAD_DISTRICT_MUN_CODES).astype("Int64")

df_mad_household = (
    df_mad_household[df_mad_household["mun_code"].notna()]
    [["mun_code", "mun_name", "district_name"] + cols_hogar]
    .copy()
    .reset_index(drop=True)
)

print(df_mad_household.shape)
display(df_mad_household.head())

In [ ]:
df_mad_household.to_csv(PROCESSED_PATH / "mad_household.csv", index=False)

print(f"Guardado: mad_household.csv  →  {df_mad_household.shape}")

## Probabilidad de Coche

In [ ]:
tables = camelot.read_pdf(str(EXTERNAL_PATH / "CAPITULO 7.pdf"), pages="6", flavor="stream")

print(f"Tablas encontradas: {len(tables)}")

df_car_raw = tables[0].df.copy()
df_car_raw.columns = df_car_raw.iloc[2]
df_car_raw = df_car_raw.iloc[3:].reset_index(drop=True)
df_car_raw.columns.name = None

if "motores" in df_car_raw.columns:
    df_car_raw = df_car_raw.rename(columns={"motores": "Ciclomotores"})

df_car_raw = df_car_raw[df_car_raw["Distrito"].notna()]
df_car_raw = df_car_raw[
    ~df_car_raw["Distrito"].astype(str).str.contains(
        "NOTAS|Fuente|Total Municipal|No consta", case=False, na=False
    )
]

numeric_cols = df_car_raw.columns.drop("Distrito")
for col in numeric_cols:
    df_car_raw[col] = pd.to_numeric(
        df_car_raw[col].astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace("", None),
        errors="coerce",
    )

df_car_raw = df_car_raw.dropna(subset=numeric_cols).reset_index(drop=True)
df_car_raw[numeric_cols] = df_car_raw[numeric_cols].astype(int)

df_mad_car = df_car_raw[["Distrito"]].copy()
df_mad_car["prob_car"] = df_car_raw["Turismos"] / df_car_raw["Habitantes"]

df_mad_car[["district_num", "district_name"]] = df_mad_car["Distrito"].str.extract(r"(\d+)\s+(.*)")
df_mad_car["mun_name"] = "Madrid"
df_mad_car["mun_code"] = df_mad_car["district_name"].str.strip().map(MAD_DISTRICT_MUN_CODES).astype("Int64")

df_mad_car = (
    df_mad_car[df_mad_car["mun_code"].notna()]
    [["mun_code", "mun_name", "district_name", "prob_car"]]
    .copy()
    .reset_index(drop=True)
)

print(df_mad_car.shape)
display(df_mad_car)

In [ ]:
df_mad_car.to_csv(PROCESSED_PATH / "mad_car_prob.csv", index=False)

print(f"Guardado: mad_car_prob.csv  →  {df_mad_car.shape}")

---
# Tasas Provinciales

## Inactividad

In [ ]:
df_inactive_raw = pd.read_csv(EXTERNAL_PATH / "prov_people_inactive.csv", encoding="latin1", sep=";")

df_inactive_raw["Total"] = (
    df_inactive_raw["Total"]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
    .astype(int)
)

df_inactive_prov = df_inactive_raw[
    ~df_inactive_raw["Provincias"].str.contains("Total", case=False, na=False)
].copy()

df_inactive_prov[["prov_code", "prov_name"]] = df_inactive_prov["Provincias"].str.extract(r"(\d+)\s+(.*)")
df_inactive_prov["prov_code"] = df_inactive_prov["prov_code"].astype(int)

df_prov_inactivity = (
    df_inactive_prov
    .pivot_table(
        index=["prov_code", "prov_name"],
        columns="Situación de inactividad",
        values="Total",
        aggfunc="sum"
    )
    .reset_index()
)

base = df_prov_inactivity["Total"]
for col in df_prov_inactivity.columns:
    if col not in ["prov_code", "prov_name", "Total"]:
        df_prov_inactivity[f"rate_{col.lower().replace(' ', '_')}"] = df_prov_inactivity[col] / base

df_prov_inactivity = df_prov_inactivity.rename(columns={
    "rate_estudiante":              "rate_student_inactive",
    "rate_incapacitado_permanente": "rate_disabled_inactive",
    "rate_jubilado_o_pensionista":  "rate_retired_inactive",
    "rate_labores_del_hogar":       "rate_household_inactive",
    "rate_otra":                    "rate_other_inactive",
})[["prov_code", "prov_name",
    "rate_student_inactive", "rate_disabled_inactive",
    "rate_retired_inactive",  "rate_household_inactive",
    "rate_other_inactive"]]

print(df_prov_inactivity.shape)
display(df_prov_inactivity.head())

## Actividad y Paro

In [ ]:
df_paro_raw = pd.read_csv(EXTERNAL_PATH / "rate_employ_paro.csv", encoding="latin1", sep=";")

df_paro_raw["Total"] = (
    df_paro_raw["Total"]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
    .div(100)
    .round(3)
)

df_paro_prov = df_paro_raw[
    ~df_paro_raw["Provincias"].str.contains("Total Nacional", case=False)
].copy()

df_paro_prov[["prov_code", "prov_name"]] = df_paro_prov["Provincias"].str.extract(r"(\d+)\s+(.*)")

df_paro_prov = df_paro_prov[
    (df_paro_prov["Sexo"]    == "Ambos sexos") &
    (df_paro_prov["Periodo"] == "2025T4")
]

df_paro_prov = (
    df_paro_prov
    .pivot_table(
        index=["prov_code", "prov_name"],
        columns="Tasas",
        values="Total",
        aggfunc="first"
    )
    .rename(columns={
        "Tasa de actividad":            "rate_activity",
        "Tasa de paro de la población": "rate_paro"
    })
    .reset_index()
)
df_paro_prov["prov_code"] = df_paro_prov["prov_code"].astype(int)

print(df_paro_prov.shape)
display(df_paro_prov.head())

## Estado Civil

In [ ]:
df_civil_raw = pd.read_csv(EXTERNAL_PATH / "PROV_estado_civil.csv", encoding="latin1", sep=";")

df_civil_raw["Total"] = (
    df_civil_raw["Total"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df_civil_prov = df_civil_raw[
    ~df_civil_raw["Provincias"].str.contains("Total Nacional", case=False, na=False)
    & (df_civil_raw["Periodo"].astype(str) == "2024")
    & (~df_civil_raw["Estado civil"].str.contains("No consta", na=False))
].copy()

df_civil_prov[["prov_code", "prov_name"]] = df_civil_prov["Provincias"].str.extract(r"(\d+)\s+(.*)")
df_civil_prov["prov_code"] = df_civil_prov["prov_code"].astype(int)

# Agregar hombres y mujeres
df_civil_agg = df_civil_prov.groupby(["prov_code", "Estado civil"], as_index=False)["Total"].sum()

df_civil_agg["prob"] = (
    df_civil_agg["Total"] / df_civil_agg.groupby("prov_code")["Total"].transform("sum")
)

df_prov_civil = (
    df_civil_agg
    .pivot_table(index="prov_code", columns="Estado civil", values="prob")
    .reset_index()
)
df_prov_civil.columns.name = None
df_prov_civil = df_prov_civil.rename(columns={
    "Soltero/a":                 "rate_soltero",
    "Casado/a":                  "rate_casado",
    "Viudo/a":                   "rate_viudo",
    "Divorciado/a o separado/a": "rate_divorciado",
})

print(df_prov_civil.shape)
display(df_prov_civil.head())

## Habitaciones

In [ ]:
df_hab_raw = pd.read_csv(EXTERNAL_PATH / "PROV_num_habitaciones.csv", encoding="latin1", sep=";")

df_hab_raw["Total"] = pd.to_numeric(
    df_hab_raw["Total"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .replace("", None),
    errors="coerce",
)

# El CSV no tiene codigo numerico en Provincias; construir lookup desde df_paro_raw
prov_name_to_code = {
    re.sub(r"^\d+\s+", "", p).strip(): int(re.match(r"^(\d+)", p).group(1))
    for p in df_paro_raw["Provincias"].dropna().unique()
    if re.match(r"^\d+", str(p))
}

df_hab_raw["prov_code"] = df_hab_raw["Provincias"].map(prov_name_to_code)

df_hab_prov = df_hab_raw[
    df_hab_raw["prov_code"].notna()
    & (df_hab_raw["Número de habitaciones de la vivienda"] != "Total")
    & (df_hab_raw["Tamaño del hogar"] == "Total")
].copy()

df_hab_prov["prov_code"] = df_hab_prov["prov_code"].astype(int)

df_hab_prov["prob"] = (
    df_hab_prov["Total"] / df_hab_prov.groupby("prov_code")["Total"].transform("sum")
)

df_prov_hab = (
    df_hab_prov
    .pivot_table(index="prov_code", columns="Número de habitaciones de la vivienda", values="prob")
    .reset_index()
)
df_prov_hab.columns.name = None
df_prov_hab = df_prov_hab.rename(columns={
    "Menos de 3 habitaciones": "rate_hab_menos3",
    "Entre 3 y 6 habitaciones": "rate_hab_3a6",
    "7 o más habitaciones":    "rate_hab_7mas",
})

print(df_prov_hab.shape)
display(df_prov_hab.head())

## Unificación

In [ ]:
df_prov_enriched = (
    df_prov_inactivity
    .merge(df_paro_prov.drop(columns=["prov_name"]), on="prov_code", how="left")
    .merge(df_prov_civil,                             on="prov_code", how="left")
    .merge(df_prov_hab,                               on="prov_code", how="left")
)

print(df_prov_enriched.shape)
display(df_prov_enriched.head())

In [ ]:
df_prov_enriched.to_csv(PROCESSED_PATH / "prov_enriched.csv", index=False)

print(f"Guardado: prov_enriched.csv  →  {df_prov_enriched.shape}")

---
# Sectores Municipales

In [ ]:
df_sector = pd.read_csv(EXTERNAL_PATH / "MUN_sector_values.csv", encoding="utf-8", sep=";", low_memory=False)

print(df_sector.shape)
display(df_sector.head())

In [ ]:
df_sector["mun_code_base"] = df_sector["mun_code"].apply(normalize_mun_code).astype("Int64")
df_sector.to_csv(PROCESSED_PATH / "sector_mun.csv", index=False)
print(f"Guardado: sector_mun.csv  -  {df_sector.shape}")

---
# Hogares Municipales

In [ ]:
df_mun_hogar_raw = pd.read_csv(
    EXTERNAL_PATH / "MUN_size_hogar_personas.csv", encoding="utf-8", sep=";"
)

# Excluir Total Nacional y la categoría Total
df_mun_hogar_raw = df_mun_hogar_raw[
    ~df_mun_hogar_raw["Municipio"].str.contains("Total Nacional", case=False, na=False)
    & (df_mun_hogar_raw["Tamaño del hogar"] != "Total (tamaño del hogar)")
].copy()

# Extraer mun_code del campo '01001 Alegría-Dulantzi'
df_mun_hogar_raw["mun_code"] = (
    df_mun_hogar_raw["Municipio"].str.extract(r"^(\d+)").astype(int)
)

df_mun_hogar_raw["Total"] = pd.to_numeric(df_mun_hogar_raw["Total"], errors="coerce")
df_mun_hogar_raw = df_mun_hogar_raw.dropna(subset=["Total"])

df_mun_hogar_raw["prob"] = (
    df_mun_hogar_raw["Total"]
    / df_mun_hogar_raw.groupby("mun_code")["Total"].transform("sum")
)

df_mun_household = (
    df_mun_hogar_raw[["mun_code", "Tamaño del hogar", "prob"]]
    .rename(columns={"Tamaño del hogar": "household_size"})
    .reset_index(drop=True)
)

print(df_mun_household.shape)
display(df_mun_household.head(10))

In [ ]:
df_mun_household.to_csv(PROCESSED_PATH / "mun_household.csv", index=False)

print(f"Guardado: mun_household.csv  →  {df_mun_household.shape}")